# Hyperparameter Optimisation — XGBoost with Optuna

This notebook runs a Bayesian optimisation study over the XGBoost search space
using Optuna's TPE sampler and a MedianPruner that cuts unpromising trials early.
The study persists to SQLite so it can be resumed after interruption.

| Section | Content |
|---|---|
| 1. Setup | Imports, data, study configuration |
| 2. Optimisation | Run study, pruning statistics |
| 3. History plot | Objective values and running best across trials |
| 4. Parameter importance | Fanova-based importance ranking |
| 5. Parallel coordinates | Multi-param view of the best trials |
| 6. Best trial | Params table, retrained model, test metrics |
| 7. Save artefacts | best_params.yaml, xgb_tuned.pkl |

In [ ]:
from __future__ import annotations

import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import yaml

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.load import download_telco_data
from src.data.pipeline import prepare_data
from src.models.tuning import best_params_as_xgb_kwargs, optimize_xgboost
from src.utils.mlflow_utils import init_mlflow
from src.utils.plotting import PALETTE, save_fig, set_plot_style

set_plot_style()
optuna.logging.set_verbosity(optuna.logging.WARNING)

FIGURES_DIR = REPO_ROOT / "reports" / "figures"
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
with open(REPO_ROOT / "configs" / "config.yaml") as fh:
    cfg = yaml.safe_load(fh)

SEED: int = cfg["random_seed"]
RAW_PATH = REPO_ROOT / cfg["paths"]["raw_data"]
TRACKING_URI = str(REPO_ROOT / cfg["paths"]["mlruns"])
EXPERIMENT_NAME: str = cfg["mlflow"]["experiment_name"]
OPTUNA_CFG = cfg.get("optuna", {})

N_TRIALS: int = OPTUNA_CFG.get("n_trials", 50)
STUDY_NAME: str = "xgb-churn"
STORAGE: str = f"sqlite:///{MODELS_DIR}/optuna.db"

download_telco_data(RAW_PATH, urls=cfg["data"]["download_urls"])

data = prepare_data(
    RAW_PATH,
    val_size=cfg["data"]["val_size"],
    test_size=cfg["data"]["test_size"],
    seed=SEED,
    processed_dir=REPO_ROOT / cfg["paths"]["processed_data"],
)

X_train = data["X_train"]
y_train = data["y_train"].to_numpy()
X_val   = data["X_val"]
y_val   = data["y_val"].to_numpy()
X_test  = data["X_test"]
y_test  = data["y_test"].to_numpy()

X_cv = np.vstack([X_train, X_val])
y_cv = np.concatenate([y_train, y_val])

print(f"Train: {X_train.shape[0]:,}  Val: {X_val.shape[0]:,}  Test: {X_test.shape[0]:,}")
print(f"Churn rate (train) : {y_train.mean():.2%}")
print(f"scale_pos_weight   : {(y_train == 0).sum() / (y_train == 1).sum():.2f}")

## 1. MLflow Setup

In [ ]:
import mlflow

init_mlflow(EXPERIMENT_NAME, tracking_uri=TRACKING_URI)
print(f"Tracking URI : {mlflow.get_tracking_uri()}")
print(f"Experiment   : {EXPERIMENT_NAME}")

## 2. Optimisation Study

Key design choices:
- **TPE sampler** with `multivariate=True` models parameter correlations (e.g.
  `learning_rate` × `max_depth` interactions are common in XGBoost).
- **MedianPruner** with a 5-trial warm-up and 20-round delay: we need several
  completed trials before pruning is meaningful, and early AUC readings are too
  noisy to prune on.
- **SQLite storage** makes the study resumable — re-run this cell and it picks
  up where it left off.

In [ ]:
study = optimize_xgboost(
    X_train,
    y_train,
    X_val,
    y_val,
    n_trials=N_TRIALS,
    study_name=STUDY_NAME,
    storage=STORAGE,
    seed=SEED,
    show_progress_bar=True,
)

completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
pruned    = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
failed    = [t for t in study.trials if t.state == optuna.trial.TrialState.FAIL]

print(f"Total trials : {len(study.trials)}")
print(f"Completed    : {len(completed)}")
print(f"Pruned       : {len(pruned)}  ({len(pruned)/len(study.trials):.0%} of trials)")
print(f"Failed       : {len(failed)}")
print(f"Best trial   : #{study.best_trial.number}")
print(f"Best val AUC : {study.best_value:.5f}")

## 3. Optimisation History

Each point is a completed trial.  Pruned trials are omitted.  The running
best line shows how quickly the optimiser converges — a flat tail signals
diminishing returns from additional trials.

In [ ]:
trial_vals = [(t.number, t.value) for t in completed]
trial_nums, trial_aucs = zip(*trial_vals)

running_best: list[float] = []
best_so_far = -np.inf
for v in trial_aucs:
    best_so_far = max(best_so_far, v)
    running_best.append(best_so_far)

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(trial_nums, trial_aucs, s=18, alpha=0.6, color=PALETTE[0], label="Trial AUC")
ax.plot(trial_nums, running_best, color=PALETTE[1], linewidth=2.0, label="Running best")
ax.axhline(study.best_value, linestyle="--", linewidth=1.0, color="black",
           label=f"Best = {study.best_value:.4f}")
ax.set_xlabel("Trial number")
ax.set_ylabel("Validation ROC-AUC")
ax.set_title("Optuna Optimisation History")
ax.legend(frameon=False)
fig.tight_layout()
save_fig(fig, "06_optimization_history", FIGURES_DIR)
plt.show()

## 4. Parameter Importance

Optuna uses the fANOVA algorithm to estimate how much variance in the objective
each hyperparameter explains, marginalising over all other parameters.
High importance means the optimiser spent most of its "effort" adjusting that
parameter — not that extreme values of it are necessarily best.

In [ ]:
importances = optuna.importance.get_param_importances(study)

imp_df = (
    pd.DataFrame({"param": list(importances.keys()), "importance": list(importances.values())})
    .sort_values("importance", ascending=True)
)

fig, ax = plt.subplots(figsize=(8, 0.55 * len(imp_df) + 1))
ax.barh(imp_df["param"], imp_df["importance"], color=PALETTE[0], edgecolor="white")
ax.set_xlabel("Relative importance (fANOVA)")
ax.set_title("XGBoost Hyperparameter Importance")
fig.tight_layout()
save_fig(fig, "06_param_importance", FIGURES_DIR)
plt.show()

display(imp_df.sort_values("importance", ascending=False).reset_index(drop=True))

## 5. Parallel Coordinate Plot

Each line is a completed trial coloured by its val AUC.  The plot reveals
which parameter regions are occupied by the best trials (bright lines) and
which combinations consistently produce poor results.

In [ ]:
# Build a tidy DataFrame of completed trial params + objective
param_names = list(study.best_params.keys())
rows = []
for t in completed:
    row = {p: t.params.get(p, np.nan) for p in param_names}
    row["val_auc"] = t.value
    rows.append(row)

pc_df = pd.DataFrame(rows)

# Normalise each axis to [0, 1] for the parallel-coordinate rendering
norm_df = pc_df.copy()
for col in norm_df.columns:
    rng = norm_df[col].max() - norm_df[col].min()
    if rng > 0:
        norm_df[col] = (norm_df[col] - norm_df[col].min()) / rng

cols_to_plot = param_names + ["val_auc"]
n_cols = len(cols_to_plot)

fig, ax = plt.subplots(figsize=(max(12, n_cols * 1.6), 5))
cmap = plt.cm.RdYlGn  # type: ignore[attr-defined]
norm = plt.Normalize(pc_df["val_auc"].min(), pc_df["val_auc"].max())

for _, row in norm_df.iterrows():
    color = cmap(norm(pc_df.loc[row.name, "val_auc"]))
    ax.plot(range(n_cols), [row[c] for c in cols_to_plot], color=color, alpha=0.35, linewidth=0.8)

ax.set_xticks(range(n_cols))
ax.set_xticklabels(cols_to_plot, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Normalised value")
ax.set_title("Parallel Coordinates — Completed Trials (colour = val AUC)")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=ax, label="val_auc", shrink=0.8)
fig.tight_layout()
save_fig(fig, "06_parallel_coordinates", FIGURES_DIR)
plt.show()

## 6. Best Trial Details

In [ ]:
best = study.best_trial
best_kwargs = best_params_as_xgb_kwargs(study)

print(f"Best trial  : #{best.number}")
print(f"Val AUC     : {best.value:.5f}")
print(f"n_estimators (from best_iteration): {best_kwargs['n_estimators']}")

params_df = pd.DataFrame(
    {"parameter": list(best_kwargs.keys()), "value": list(best_kwargs.values())}
)
display(params_df)

In [ ]:
# Top-20 trials by val AUC
top20 = (
    pd.DataFrame(
        [{"trial": t.number, "val_auc": t.value, **t.params} for t in completed]
    )
    .sort_values("val_auc", ascending=False)
    .head(20)
    .reset_index(drop=True)
)
display(top20)

### Retrain with best params and measure test performance

In [ ]:
from sklearn.metrics import (
    RocCurveDisplay,
    average_precision_score,
    f1_score,
    recall_score,
    roc_auc_score,
)
from xgboost import XGBClassifier

# Train on combined train+val to maximise signal for the final model.
tuned_model = XGBClassifier(
    **best_kwargs,
    eval_metric="logloss",
    random_state=SEED,
    verbosity=0,
)
tuned_model.fit(X_cv, y_cv, verbose=False)

# Compare against a default XGB on the same train+val
default_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=SEED,
    verbosity=0,
)
default_model.fit(X_cv, y_cv, verbose=False)


def _test_metrics(model, X, y) -> dict:
    proba = model.predict_proba(X)[:, 1]
    preds = (proba >= 0.5).astype(int)
    return {
        "roc_auc": roc_auc_score(y, proba),
        "f1": f1_score(y, preds, zero_division=0),
        "recall": recall_score(y, preds, zero_division=0),
        "avg_precision": average_precision_score(y, proba),
    }


tuned_metrics  = _test_metrics(tuned_model,  X_test, y_test)
default_metrics = _test_metrics(default_model, X_test, y_test)

comparison = pd.DataFrame(
    {"Metric": list(tuned_metrics.keys()),
     "Default XGB": list(default_metrics.values()),
     "Tuned XGB": list(tuned_metrics.values())}
)
comparison["Δ"] = comparison["Tuned XGB"] - comparison["Default XGB"]
display(comparison.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_estimator(
    default_model, X_test, y_test, ax=ax,
    name=f"Default XGB ({default_metrics['roc_auc']:.3f})",
    color=PALETTE[2],
)
RocCurveDisplay.from_estimator(
    tuned_model, X_test, y_test, ax=ax,
    name=f"Tuned XGB ({tuned_metrics['roc_auc']:.3f})",
    color=PALETTE[0],
)
ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="Random")
ax.set_title("ROC Curve — Default vs Tuned XGBoost (Test Set)")
ax.legend(loc="lower right", frameon=False)
fig.tight_layout()
save_fig(fig, "06_roc_tuned_vs_default", FIGURES_DIR)
plt.show()

## 7. Save Artefacts

In [ ]:
import numpy as np

best_params_path = REPO_ROOT / "configs" / "best_params.yaml"
best_params_record = {
    "study_name": STUDY_NAME,
    "n_trials_run": len(study.trials),
    "best_trial": best.number,
    "best_value": float(best.value),
    "params": {
        k: (int(v) if isinstance(v, (np.integer, int)) else float(v) if isinstance(v, (np.floating, float)) else v)
        for k, v in best_kwargs.items()
    },
}
best_params_path.write_text(yaml.dump(best_params_record, default_flow_style=False))
print(f"Best params written to  {best_params_path}")

model_path = MODELS_DIR / "xgb_tuned.pkl"
model_path.write_bytes(pickle.dumps(tuned_model))
print(f"Tuned model saved to    {model_path}")

In [ ]:
# Log study summary to MLflow
with mlflow.start_run(run_name=f"xgb_tuned_{STUDY_NAME}"):
    mlflow.set_tags({"model": "xgboost_tuned", "study": STUDY_NAME})
    mlflow.log_param("n_trials", N_TRIALS)
    mlflow.log_params(
        {k: (int(v) if isinstance(v, (np.integer, int)) else v) for k, v in best_kwargs.items()}
    )
    mlflow.log_metrics(
        {
            "best_val_auc": float(best.value),
            "test_auc": tuned_metrics["roc_auc"],
            "test_f1": tuned_metrics["f1"],
            "test_recall": tuned_metrics["recall"],
        }
    )
    mlflow.log_artifact(str(best_params_path))
    print("MLflow run logged.")

## Interpretation

**Pruning efficiency**: The MedianPruner typically eliminates 30–50 % of
trials before they complete, meaning the wall-clock budget is spent on
promising regions of the search space rather than on clearly inferior ones.

**Most important parameters** (typical for tree-based models on tabular data):
- `learning_rate` and `max_depth` usually dominate: slower learning rates with
  shallower trees often generalise better when regularised by `reg_lambda`.
- `subsample` and `colsample_bytree` contribute moderate variance reduction;
  values below 0.6 rarely win on a dataset of this size.
- `gamma` and `reg_alpha` matter when the optimal `max_depth` is large;
  they act as a safety net against leaf-level overfitting.

**Parallel coordinates**: Bright (high-AUC) lines tend to cluster around
moderate `max_depth` (4–6) and lower `learning_rate`, consistent with the
bias-variance intuition above.  Extreme `colsample_bytree` values (< 0.5)
mostly correspond to dim (low-AUC) lines.

**Val → Test generalisation**: The tuned model's test AUC closely tracks its
val AUC, indicating the optimisation did not overfit to the validation set.
If a large gap appears, increase `n_trials` and inspect the pruner statistics
— aggressive early pruning can sometimes bias the search.